# run_e19
CPU E19 analysis worker. Run cells 2, 3 and 4 in order. Configure only cell 3. Select one `MODEL` (`qwen` or `gemma`) and supply the prepared Stage 4 plan and explicit priority-0 midpoint SCREEN + DEV-C prediction references. Optionally restrict `SOURCES` (e.g. `['en']`) for a representative measurement run and pass its completed reference in `RESUME_REFS` when running all four sources.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage4')
CODE_REF = 'phase1'  # Resolved once below; set this to the approved Stage 4 commit to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
MODEL = 'qwen'  # 'qwen' or 'gemma'
STAGE4_PLAN = PHASE1_ROOT / 'REPLACE_STAGE4_PLAN/_COMPLETE.json'
PREDICTION_REFS_FILE = PHASE1_ROOT / 'REPLACE_E19_PREDICTION_REFS.json'
SOURCES = None  # None for all four sources ('en','hi','bn','te'), or e.g. ['en'] for representative run.
RESUME_REFS = []  # Explicit completed E19 analysis references ({'path': ..., 'sha256': ...}) to reuse.
OUTPUT_INDEX = None  # Optional distinct mounted JSON path for this model's handoff.


In [ ]:
import json
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Stage 4 checkout.')
for path in (STAGE4_PLAN, PREDICTION_REFS_FILE):
    if not path.is_file():
        raise RuntimeError(f'Set the explicit mounted artifact in cell 3: {path}')
command = [
    sys.executable, '-u', '-m', 'src.stage4.run_e19',
    '--phase1-root', str(PHASE1_ROOT),
    '--manifest', str(STAGE4_PLAN),
    '--model', MODEL,
    '--predictions', str(PREDICTION_REFS_FILE),
    '--storage-scope', 'persistent-drive',
]
if SOURCES is not None:
    command += ['--sources', ','.join(SOURCES)]
for ref in RESUME_REFS:
    command += ['--resume', json.dumps(ref) if isinstance(ref, dict) else str(ref)]
if OUTPUT_INDEX is not None:
    command += ['--output-index', str(OUTPUT_INDEX)]
process = subprocess.Popen(command, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    raise
if process.wait() != 0:
    raise RuntimeError('E19 worker blocked/failed. Preserve streamed references before resuming.')
